# About dataset
Context This is a small subset of dataset of Book reviews from Amazon Kindle Store

Content 5-core dataset of product reviews from Amazon Kindle Store category from May to July 2014. Contains total of 982619 entries. Each reviewer has at least 5 reviews and each product has at least 5 reviews in this dataset. Columns

asin - ID of the product, like B000FA64PK

helpful - helpfulness rating of the review - example: 2/3.

overall - rating of the product.

reviewText - text of the review (heading).

reviewTime - time of the review (raw).

reviewerID - ID of the reviewer, like A3SPTOKDG7WBLN

reviewerName - name of the reviewer.

summary - summary of the review (description).

unixReviewTime - unix timestamp.

Acknowledgements This dataset is taken from Amazon product data, Julian McAuley's, UCSD website. http://jmcauley.ucsd.edu/data/amazon/

In [2]:
import pandas as pd
df = pd.read_csv('all_kindle_review.csv')

In [3]:
df.head()

,Unnamed: 0.1,Unnamed: 0,asin,helpful,rating,reviewText,reviewTime,reviewerID,reviewerName,summary,unixReviewTime
0,0,11539,B0033UV8HI,"[8, 10]",3,"Jace Rankin may be short, but he's nothing to ...","09 2, 2010",A3HHXRELK8BHQG,Ridley,Entertaining But Average,1283385600
1,1,5957,B002HJV4DE,"[1, 1]",5,Great short read. I didn't want to put it dow...,"10 8, 2013",A2RGNZ0TRF578I,Holly Butler,Terrific menage scenes!,1381190400
2,2,9146,B002ZG96I4,"[0, 0]",3,I'll start by saying this is the first of four...,"04 11, 2014",A3S0H2HV6U1I7F,Merissa,Snapdragon Alley,1397174400
3,3,7038,B002QHWOEU,"[1, 3]",3,Aggie is Angela Lansbury who carries pocketboo...,"07 5, 2014",AC4OQW3GZ919J,Cleargrace,very light murder cozy,1404518400
4,4,1776,B001A06VJ8,"[0, 1]",4,I did not expect this type of book to be in li...,"12 31, 2012",A3C9V987IQHOQD,Rjostler,Book,1356912000


In [4]:
df = df [['reviewText', 'rating']]
df.head()

,reviewText,rating
0,"Jace Rankin may be short, but he's nothing to ...",3
1,Great short read. I didn't want to put it dow...,5
2,I'll start by saying this is the first of four...,3
3,Aggie is Angela Lansbury who carries pocketboo...,3
4,I did not expect this type of book to be in li...,4


In [5]:
df.shape

(12000, 2)

In [6]:
df.isnull().sum()

reviewText    0
rating        0
dtype: int64

In [7]:
df['rating'].unique()

array([3, 5, 4, 2, 1])

In [8]:
df['rating'].value_counts()

rating
5    3000
4    3000
3    2000
2    2000
1    2000
Name: count, dtype: int64

# Preprocessing and Cleaning

In [9]:
# Positive review is 1 and negative is 0
df['rating'] = df['rating'].apply(lambda x : 0 if x < 3 else 1)

In [10]:
df.sample(5)

,reviewText,rating
872,"Characters: Atlas, the Titan god of strength ...",0
11284,This was definitely well written. I was most i...,1
3166,"its not the worse, but it had a childish feel ...",0
6683,This is a very short novella . Lord Alexander ...,1
1601,OMG this book was sooo bad. I loved the Maste...,0


In [11]:
df['rating'].unique()

array([1, 0])

In [12]:
df['rating'].value_counts()

rating
1    8000
0    4000
Name: count, dtype: int64

In [13]:
# Lower all the cases
df['reviewText'] = df['reviewText'].str.lower()

In [14]:
df['reviewText'].head()

0    jace rankin may be short, but he's nothing to ...
1    great short read.  i didn't want to put it dow...
2    i'll start by saying this is the first of four...
3    aggie is angela lansbury who carries pocketboo...
4    i did not expect this type of book to be in li...
Name: reviewText, dtype: object

In [15]:
! pip install beautifulsoup4
! pip install lxml

In [16]:
import re
import nltk
from nltk.corpus import stopwords
from bs4 import BeautifulSoup
# Removing special characters
df['reviewText'] = df['reviewText'].apply(lambda x: re.sub('[^a-z A-Z 0-9-]+', '', x))

# Removing the stopwords
df['reviewText'] = df['reviewText'].apply(lambda x: ' '.join([y for y in x.split() if y not in stopwords.words('english')]))

# Removing url
df['reviewText'] = df['reviewText'].apply(lambda x: re.sub(r'(http|https|ftp|ssh)://(\w_-]+(?:(?:\.[\w_-]+)+))([\w.,@?^=%&:/~+#-]*[\w@?^=%&/~+#-])?', '' , str(x)))

# Removing HTML tags
df['reviewText'] = df['reviewText'].apply(lambda x: BeautifulSoup(x, 'lxml').get_text())

# Removing any additional spaces
df['reviewText'] = df['reviewText'].apply(lambda x: '  '.join(x.split()))

KeyboardInterrupt: 

In [17]:
# works same as the upper code but faster

import re
from nltk.corpus import stopwords

# 1. Pre-compile Regex (Significant speed boost)
URL_PATTERN = re.compile(r'(http|https|ftp|ssh)://[\w\-_]+(?:\.[\w\-_]+)+(?:[\w\.,@?^=%&:/~+#-]*[\w@?^=%&/~+#-])?')
TAG_PATTERN = re.compile(r'<[^>]*>')
SPECIAL_CHARS = re.compile(r'[^a-zA-Z0-9\s-]')

# 2. Use a SET for stopwords (Lookups are O(1) vs O(n) for lists)
STOPWORDS = set(stopwords.words('english'))

def fast_clean(text):
    # Convert to string and lowercase once
    text = str(text).lower()
    
    # Remove HTML tags (Regex is much faster than BeautifulSoup for simple tags)
    text = TAG_PATTERN.sub('', text)
    
    # Remove URLs
    text = URL_PATTERN.sub('', text)
    
    # Remove Special Characters (but keep spaces and hyphens)
    text = SPECIAL_CHARS.sub('', text)
    
    # Split, remove stopwords, and fix spaces in one go
    words = [w for w in text.split() if w not in STOPWORDS]
    
    return " ".join(words)

# 3. Apply once
df['reviewText'] = df['reviewText'].apply(fast_clean)

In [18]:
df['reviewText']

0        jace rankin may short hes nothing mess man hau...
1        great short read didnt want put read one sitti...
2        ill start saying first four books wasnt expect...
3        aggie angela lansbury carries pocketbooks inst...
4        expect type book library pleased find price right
                               ...                        
11995    valentine cupid vampire- jena ian another vamp...
11996    read seven books series apocalypticadventure o...
11997    book really wasnt cuppa situation man capturin...
11998    tried use charge kindle didnt even register ch...
11999    taking instruction look often hidden world sex...
Name: reviewText, Length: 12000, dtype: object

In [19]:
# Lemmatization
from nltk.stem import WordNetLemmatizer
lemmatizer = WordNetLemmatizer()

In [20]:
def lemmatize_words(text):
    return " ".join(lemmatizer.lemmatize(word) for word in text.split())

In [21]:
df['reviewText'] = df['reviewText'].apply(lambda x: lemmatize_words(x))

In [22]:
df.head()

,reviewText,rating
0,jace rankin may short he nothing mess man haul...,1
1,great short read didnt want put read one sitti...,1
2,ill start saying first four book wasnt expecti...,1
3,aggie angela lansbury carry pocketbook instead...,1
4,expect type book library pleased find price right,1


In [23]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split (df['reviewText'], df['rating'], test_size= 0.20, random_state= 42)

In [24]:
from sklearn.feature_extraction.text import TfidfVectorizer
# Run this before the vectorizer to ensure no non-string objects exist
X_train = X_train.astype(str)
X_test = X_test.astype(str)

tfidf = TfidfVectorizer()
X_train_tfidf = tfidf.fit_transform(X_train).toarray()
X_test_tfidf = tfidf.transform(X_test).toarray()

In [25]:
from sklearn.feature_extraction.text import CountVectorizer
bow = CountVectorizer()
X_train_bow = bow.fit_transform(X_train).toarray()
X_test_bow = bow.transform(X_test).toarray()

In [26]:
from sklearn.naive_bayes import GaussianNB
nb_model_bow = GaussianNB().fit(X_train_bow, y_train)
nb_model_tfidf = GaussianNB().fit(X_train_tfidf, y_train)

In [27]:
y_pred_bow = nb_model_bow.predict(X_test_bow)
y_pred_tfidf = nb_model_bow.predict(X_test_tfidf)

In [28]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

print("Classification Score bow: ", classification_report(y_test, y_pred_bow))
print("Accuracy Score bow: ", accuracy_score(y_test, y_pred_bow))
print("Confusion matrix Score bow: ", confusion_matrix(y_test, y_pred_bow))

print("Classification Score tfidf: ", classification_report(y_test, y_pred_tfidf))
print("Accuracy Score tfidf: ", accuracy_score(y_test, y_pred_tfidf))
print("Confusion matrix Score tfidf: ", confusion_matrix(y_test, y_pred_tfidf))

Classification Score bow:                precision    recall  f1-score   support

           0       0.41      0.62      0.49       803
           1       0.74      0.55      0.63      1597

    accuracy                           0.57      2400
   macro avg       0.58      0.59      0.56      2400
weighted avg       0.63      0.57      0.59      2400

Accuracy Score bow:  0.5745833333333333
Confusion matrix Score bow:  [[499 304]
 [717 880]]
Classification Score tfidf:                precision    recall  f1-score   support

           0       0.41      0.61      0.49       803
           1       0.74      0.56      0.63      1597

    accuracy                           0.57      2400
   macro avg       0.57      0.58      0.56      2400
weighted avg       0.63      0.57      0.59      2400

Accuracy Score tfidf:  0.57375
Confusion matrix Score tfidf:  [[490 313]
 [710 887]]


In [29]:
import gensim
from gensim.models import Word2Vec
import gensim.downloader as api

In [30]:
wv = api.load('word2vec-google-news-300')

In [31]:
X_train.head()

9182     looking forward book came double space every p...
11091    already owned book spouse forgot already part ...
6428     cool forgot request rate came make mine unreli...
288      short short story basically scene party one ni...
2626     secret service agent secrests even longer serv...
Name: reviewText, dtype: object

In [32]:
from nltk import sent_tokenize
from gensim.utils import simple_preprocess
words = []
for sent in df['reviewText']:
    sent_tokens = sent_tokenize(sent)
    for sent in sent_tokens:
        words.append(simple_preprocess(sent))

In [33]:
words

[['jace',
  'rankin',
  'may',
  'short',
  'he',
  'nothing',
  'mess',
  'man',
  'hauled',
  'saloon',
  'undertaker',
  'know',
  'he',
  'famous',
  'bounty',
  'hunter',
  'oregon',
  'shot',
  'man',
  'saloon',
  'finished',
  'year',
  'long',
  'quest',
  'avenge',
  'sister',
  'murder',
  'trying',
  'figure',
  'next',
  'snotty',
  'nosed',
  'farm',
  'boy',
  'rescued',
  'gang',
  'bully',
  'offer',
  'money',
  'kill',
  'man',
  'forced',
  'ranch',
  'reluctantly',
  'agrees',
  'bring',
  'man',
  'justice',
  'kill',
  'outright',
  'first',
  'need',
  'tell',
  'sister',
  'widower',
  'newskyla',
  'kyle',
  'springer',
  'bailey',
  'riding',
  'trail',
  'sleeping',
  'ground',
  'past',
  'month',
  'trying',
  'find',
  'jace',
  'want',
  'revenge',
  'man',
  'killed',
  'husband',
  'took',
  'ranch',
  'amongst',
  'crime',
  'shes',
  'keen',
  'detour',
  'jace',
  'want',
  'take',
  'realizes',
  'shes',
  'option',
  'hide',
  'behind',
  'boy',
 

In [34]:
model = gensim.models.Word2Vec(words)

In [35]:
model.wv.index_to_key

['book',
 'story',
 'read',
 'one',
 'character',
 'like',
 'good',
 'would',
 'really',
 'love',
 'time',
 'get',
 'author',
 'reading',
 'series',
 'well',
 'much',
 'first',
 'even',
 'didnt',
 'short',
 'know',
 'way',
 'great',
 'could',
 'make',
 'sex',
 'little',
 'dont',
 'two',
 'thing',
 'want',
 'think',
 'find',
 'plot',
 'romance',
 'also',
 'end',
 'life',
 'im',
 'see',
 'enjoyed',
 'go',
 'scene',
 'never',
 'written',
 'take',
 'woman',
 'many',
 'lot',
 'kindle',
 'year',
 'say',
 'thought',
 'work',
 'bit',
 'found',
 'going',
 'give',
 'interesting',
 'liked',
 'writing',
 'novel',
 'loved',
 'another',
 'feel',
 'better',
 'got',
 'come',
 'man',
 'hot',
 'still',
 'back',
 'enough',
 'though',
 'people',
 'star',
 'reader',
 'made',
 'something',
 'review',
 'part',
 'friend',
 'page',
 'cant',
 'bad',
 'world',
 'need',
 'free',
 'new',
 'keep',
 'wasnt',
 'doesnt',
 'relationship',
 'enjoy',
 'recommend',
 'together',
 'next',
 'start',
 'felt',
 'best',
 'put',

In [ ]:
wv.similar_by_key('know')

In [ ]:
y_pred = model.wv.predict(X_test)

NameError: name 'model' is not defined